# Team Bahlil Notebook

1. Wisnu Daarmawan (Leader)
2. Clarosa Canara
3. Annisa Wafa Jayyida

## SECTION 1 - Environment Setup & Library Import

In [ ]:
import os
import sys
import glob
import math
import random
import warnings
import subprocess
from pathlib import Path
from typing import Optional

warnings.filterwarnings("ignore")

# Global Config
TARGET = "property_organic_content"
ID_COL = "sample_id"
N_SPLITS = 5
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 200
FAST_MODE = False

# Training size control
MAIN_N_ESTIMATORS = 500 if FAST_MODE else 3000
BASELINE_N_ESTIMATORS = 300 if FAST_MODE else 1000


def seed_everything(seed: int = RANDOM_STATE) -> None:
    """Set seeds for reproducibility."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    import numpy as np
    np.random.seed(seed)

seed_everything(RANDOM_STATE)


def install_if_missing(package_name: str, import_name: Optional[str] = None) -> None:
    """Install a package if import fails. Useful for Colab/local notebooks."""
    import importlib
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

# Required stack
install_if_missing("catboost")
install_if_missing("lightgbm")
install_if_missing("xgboost")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
from lightgbm import LGBMRegressor
import xgboost as xgb
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


def rmse(y_true, y_pred) -> float:
    """Root Mean Squared Error, version-safe across sklearn releases."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print("Environment ready")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"lightgbm: {lgb.__version__}")
print(f"xgboost: {xgb.__version__}")

## SECTION 2 - Load Data

Di bagian ini, notebook dibuat supaya bisa jalan di berbagai tempat.

Terdapat pengecekan di awal supaya kalau ada file yang salah, kurang, atau kolom pentingnya nggak ada, akan nmuncul error.

In [ ]:
def find_csv_file(filename: str) -> Optional[str]:
    """Find CSV file across common Kaggle, Colab, local, and sandbox paths."""
    search_patterns = [
        filename,
        f"./{filename}",
        f"/content/{filename}",
        f"/kaggle/working/{filename}",
        f"/kaggle/input/**/{filename}",
    ]
    matches = []
    for pattern in search_patterns:
        matches.extend(glob.glob(pattern, recursive=True))
    matches = [m for m in matches if os.path.isfile(m)]
    if not matches:
        return None
    # Prefer local/current files, then /content, then /kaggle/working, then /kaggle/input.
    priority = {".": 0, "/content": 1, "/kaggle/working": 2, "/kaggle/input": 3}
    def score(path: str):
        p = os.path.abspath(path)
        for key, value in priority.items():
            if p == os.path.abspath(filename) or p.startswith(os.path.abspath(key)):
                return value
        return 99
    return sorted(set(matches), key=score)[0]


def maybe_colab_upload(required_files: list[str]) -> None:
    """If files are missing and running in Colab, ask user to upload them."""
    missing = [f for f in required_files if find_csv_file(f) is None]
    if not missing:
        return
    try:
        from google.colab import files
        print("Missing files detected:", missing)
        print("Please upload train.csv, test.csv, and sample_submission.csv")
        uploaded = files.upload()
        print("Uploaded:", list(uploaded.keys()))
    except Exception as exc:
        print("Not running in Colab or upload is unavailable.")
        print("Missing files:", missing)
        raise FileNotFoundError(f"Required files not found: {missing}") from exc


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load train, test, and sample submission with strict schema validation."""
    required_files = ["train.csv", "test.csv", "sample_submission.csv"]
    maybe_colab_upload(required_files)

    paths = {name: find_csv_file(name) for name in required_files}
    if any(v is None for v in paths.values()):
        raise FileNotFoundError(f"Could not locate all required files. Found paths: {paths}")

    print("Resolved paths:")
    for k, v in paths.items():
        print(f"- {k}: {v}")

    train = pd.read_csv(paths["train.csv"])
    test = pd.read_csv(paths["test.csv"])
    sample_submission = pd.read_csv(paths["sample_submission.csv"])

    # Strict validation
    assert TARGET in train.columns, f"Train file must contain target column: {TARGET}"
    assert ID_COL in train.columns, f"Train file must contain ID column: {ID_COL}"
    assert ID_COL in test.columns, f"Test file must contain ID column: {ID_COL}"
    assert ID_COL in sample_submission.columns, f"Sample submission must contain ID column: {ID_COL}"
    assert TARGET in sample_submission.columns, f"Sample submission must contain target column: {TARGET}"
    assert len(test) == len(sample_submission), "Test and sample_submission must have the same number of rows"
    assert train[ID_COL].is_unique, f"Train {ID_COL} must be unique"
    assert test[ID_COL].is_unique, f"Test {ID_COL} must be unique"
    assert sample_submission[ID_COL].is_unique, f"Sample submission {ID_COL} must be unique"

    # Warn, do not fail, if order differs. Final submission will be merged onto sample_submission order.
    if not test[ID_COL].equals(sample_submission[ID_COL]):
        print("Warning: test ID order differs from sample_submission. Final output will follow sample_submission order.")

    print("\nShapes:")
    print(f"train: {train.shape}")
    print(f"test: {test.shape}")
    print(f"sample_submission: {sample_submission.shape}")

    return train, test, sample_submission

train_raw, test_raw, sample_submission = load_data()

print("\nTrain head:")
display(train_raw.head())
print("\nTest head:")
display(test_raw.head())
print("\nSample submission head:")
display(sample_submission.head())

## SECTION 3 - Initial Data Audit

Di bagian ini akan mengecek dataset shape, data types, duplicated rows, duplicated IDs, missing values, constant columns, high-missing columns, and categorical/numerical feature groups.

In [ ]:
def get_feature_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in [ID_COL, TARGET]]


def missing_summary(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    """Return side-by-side missing value summary for train and test."""
    all_cols = sorted(set(train.columns).union(test.columns))
    rows = []
    for col in all_cols:
        tr_miss = train[col].isna().sum() if col in train.columns else np.nan
        te_miss = test[col].isna().sum() if col in test.columns else np.nan
        rows.append({
            "column": col,
            "train_missing_count": tr_miss,
            "train_missing_pct": tr_miss / len(train) if col in train.columns else np.nan,
            "test_missing_count": te_miss,
            "test_missing_pct": te_miss / len(test) if col in test.columns else np.nan,
        })
    return pd.DataFrame(rows).sort_values("train_missing_pct", ascending=False)


def audit_dataframe(df: pd.DataFrame, name: str, has_target: bool = True) -> dict:
    """Print and return key audit findings."""
    feature_cols = get_feature_columns(df)
    categorical_cols = df[feature_cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
    duplicated_rows = int(df.duplicated().sum())
    duplicated_ids = int(df[ID_COL].duplicated().sum()) if ID_COL in df.columns else None

    print(f"===== {name} audit =====")
    print(f"Shape: {df.shape}")
    print(f"Duplicated rows: {duplicated_rows}")
    print(f"Duplicated {ID_COL}: {duplicated_ids}")
    print(f"Categorical features: {len(categorical_cols)}")
    print(f"Numerical features: {len(numerical_cols)}")
    print(f"Constant feature candidates: {constant_cols}")

    return {
        "feature_cols": feature_cols,
        "categorical_cols": categorical_cols,
        "numerical_cols": numerical_cols,
        "constant_cols": constant_cols,
        "duplicated_rows": duplicated_rows,
        "duplicated_ids": duplicated_ids,
    }

train_audit = audit_dataframe(train_raw, "train", has_target=True)
test_audit = audit_dataframe(test_raw, "test", has_target=False)

print("\nData types:")
display(train_raw.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))

missing_df = missing_summary(train_raw, test_raw)
print("\nMissing value summary:")
display(missing_df)

high_missing_cols = missing_df.loc[missing_df["train_missing_pct"].fillna(0) >= 0.40, "column"].tolist()
high_missing_cols = [c for c in high_missing_cols if c not in [TARGET]]
print("\nHigh-missing feature candidates >= 40% missing:")
print(high_missing_cols)

constant_cols = sorted(set(train_audit["constant_cols"]).intersection(set(test_audit["constant_cols"])))
print("\nConstant columns in both train and test, safe candidates to drop:")
print(constant_cols)

categorical_cols_raw = train_audit["categorical_cols"]
numerical_cols_raw = train_audit["numerical_cols"]
print("\nRaw categorical columns:", categorical_cols_raw)
print("Raw numerical columns:", numerical_cols_raw)

## SECTION 4 - Exploratory Data Analysis

Difokuskan pada target distribution, missingness, numeric correlations, categorical relationships, and train-test distribution checks.

In [ ]:
def plot_target_distribution(y: pd.Series) -> None:
    print("Target summary:")
    display(y.describe().to_frame().T)
    print(f"Skewness: {y.skew():.4f}")

    plt.figure(figsize=(8, 4))
    plt.hist(y.dropna(), bins=50)
    plt.title("Target Distribution")
    plt.xlabel(TARGET)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 2.5))
    plt.boxplot(y.dropna(), vert=False)
    plt.title("Target Boxplot")
    plt.xlabel(TARGET)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(np.log1p(y.dropna()), bins=50)
    plt.title("log1p(Target) Distribution")
    plt.xlabel(f"log1p({TARGET})")
    plt.ylabel("Frequency")
    plt.show()

plot_target_distribution(train_raw[TARGET])

In [ ]:
# Categorical cardinality and target summaries
categorical_cardinality = []
for c in categorical_cols_raw:
    categorical_cardinality.append({
        "column": c,
        "n_unique_train": train_raw[c].nunique(dropna=False),
        "n_unique_test": test_raw[c].nunique(dropna=False) if c in test_raw.columns else np.nan,
        "top_train": train_raw[c].value_counts(dropna=False).index[0] if c in train_raw.columns else np.nan,
    })
cat_cardinality_df = pd.DataFrame(categorical_cardinality).sort_values("n_unique_train", ascending=False)
print("Categorical cardinality:")
display(cat_cardinality_df)

important_cat_cols = [
    "source_id", "geo_zone_macro", "geo_zone_meso", "geo_zone_micro",
    "biome", "land_cover_type", "parent_rock_type", "sampling_strategy",
    "has_band_A_spectrum", "has_band_B_spectrum", "sampling_depth_cm",
]
important_cat_cols = [c for c in important_cat_cols if c in train_raw.columns]

for c in important_cat_cols:
    summary = train_raw.groupby(c, dropna=False)[TARGET].agg(["count", "mean", "median", "std"]).sort_values("median", ascending=False)
    print(f"\nTarget by {c}:")
    display(summary.head(20))

In [ ]:
# Numerical correlation with target
num_cols_for_corr = [c for c in numerical_cols_raw if c != TARGET and c in train_raw.columns]
corr_with_target = train_raw[num_cols_for_corr + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
print("Top absolute correlations with target:")
display(corr_with_target.head(30).to_frame("corr_with_target"))

# Compact correlation heatmap for top numeric features
heatmap_cols = corr_with_target.head(20).index.tolist() + [TARGET]
if len(heatmap_cols) > 1:
    corr_matrix = train_raw[heatmap_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    plt.imshow(corr_matrix, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
    plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
    plt.title("Correlation Heatmap - Top Numeric Features")
    plt.tight_layout()
    plt.show()

In [ ]:
# Missing value visualization and train & test comparison
missing_plot_df = missing_df[(missing_df["column"] != TARGET)].copy()
missing_plot_df = missing_plot_df.sort_values("train_missing_pct", ascending=False).head(30)

plt.figure(figsize=(10, 5))
plt.bar(missing_plot_df["column"], missing_plot_df["train_missing_pct"])
plt.xticks(rotation=90)
plt.ylabel("Train Missing Percentage")
plt.title("Top Missing Features in Train")
plt.tight_layout()
plt.show()

print("Train-test missing percentage comparison:")
display(missing_df.sort_values("train_missing_pct", ascending=False).head(40))

# Categorical train and test distribution comparison using total variation distance
cat_shift_rows = []
for c in important_cat_cols:
    train_dist = train_raw[c].astype(str).fillna("Missing").value_counts(normalize=True, dropna=False)
    test_dist = test_raw[c].astype(str).fillna("Missing").value_counts(normalize=True, dropna=False)
    cats = sorted(set(train_dist.index).union(set(test_dist.index)))
    tvd = 0.5 * sum(abs(train_dist.get(cat, 0) - test_dist.get(cat, 0)) for cat in cats)
    cat_shift_rows.append({"column": c, "train_unique": len(train_dist), "test_unique": len(test_dist), "total_variation_distance": tvd})
cat_shift_df = pd.DataFrame(cat_shift_rows).sort_values("total_variation_distance", ascending=False)
print("Categorical train-test distribution shift summary:")
display(cat_shift_df)

## SECTION 5 - Research Evidence Preparation

This section prepares tables and charts needed to answer the research questions later. It does **not** invent final answers. Every final answer should refer back to EDA, statistics, residual analysis, feature importance, and experiment results.

In [ ]:
research_assets = {}

def grouped_target_summary(df: pd.DataFrame, group_cols: list[str], min_count: int = 10) -> pd.DataFrame:
    available = [c for c in group_cols if c in df.columns]
    if not available:
        return pd.DataFrame()
    out = (
        df.groupby(available, dropna=False)[TARGET]
        .agg(count="count", mean="mean", median="median", std="std", min="min", max="max")
        .reset_index()
    )
    return out[out["count"] >= min_count].sort_values("median", ascending=False)

# Geographic distribution evidence
for c in ["geo_zone_macro", "geo_zone_meso", "geo_zone_micro"]:
    if c in train_raw.columns:
        research_assets[f"target_by_{c}"] = grouped_target_summary(train_raw, [c], min_count=5)
        print(f"Target distribution evidence by {c}:")
        display(research_assets[f"target_by_{c}"].head(20))

# Ecosystem and land cover evidence
for c in ["biome", "land_cover_type", "parent_rock_type"]:
    if c in train_raw.columns:
        research_assets[f"target_by_{c}"] = grouped_target_summary(train_raw, [c], min_count=5)
        print(f"Target distribution evidence by {c}:")
        display(research_assets[f"target_by_{c}"].head(20))

# Low acidity + low CEC evidence
if {"property_acidity_index", "cation_exchange_capacity"}.issubset(train_raw.columns):
    acidity_q25 = train_raw["property_acidity_index"].quantile(0.25)
    cec_mean = train_raw["cation_exchange_capacity"].mean()
    low_acidity_low_cec = train_raw[
        (train_raw["property_acidity_index"] < acidity_q25) &
        (train_raw["cation_exchange_capacity"] < cec_mean)
    ].copy()
    research_assets["low_acidity_low_cec"] = low_acidity_low_cec
    print(f"Low acidity threshold Q25: {acidity_q25:.4f}")
    print(f"Low CEC threshold mean: {cec_mean:.4f}")
    print(f"Filtered rows: {len(low_acidity_low_cec)}")
    for c in ["biome", "land_cover_type", "geo_zone_macro"]:
        if c in low_acidity_low_cec.columns and len(low_acidity_low_cec) > 0:
            print(f"Low acidity + low CEC target summary by {c}:")
            display(grouped_target_summary(low_acidity_low_cec, [c], min_count=3).head(20))

# Outlier combinations: land_cover_type + geo_zone_macro
if {"land_cover_type", "geo_zone_macro"}.issubset(train_raw.columns):
    combo = grouped_target_summary(train_raw, ["land_cover_type", "geo_zone_macro"], min_count=10)
    if not combo.empty:
        combo_mean = combo["mean"].mean()
        combo_std = combo["mean"].std(ddof=0)
        combo["z_score_mean"] = (combo["mean"] - combo_mean) / combo_std if combo_std > 0 else 0
        combo["is_outlier_abs_z_gt_2"] = combo["z_score_mean"].abs() > 2
        research_assets["land_cover_geo_outliers"] = combo.sort_values("z_score_mean", key=lambda s: s.abs(), ascending=False)
        print("Outlier evidence for land_cover_type + geo_zone_macro:")
        display(research_assets["land_cover_geo_outliers"].head(30))

# High-correlation pairs evidence
numeric_for_corr = train_raw[get_feature_columns(train_raw) + [TARGET]].select_dtypes(include=[np.number])
corr = numeric_for_corr.corr(numeric_only=True).abs()
pairs = []
cols = corr.columns.tolist()
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        val = corr.loc[c1, c2]
        if pd.notna(val) and val >= 0.80:
            pairs.append({"feature_1": c1, "feature_2": c2, "abs_corr": val})
high_corr_pairs = pd.DataFrame(pairs).sort_values("abs_corr", ascending=False) if pairs else pd.DataFrame(columns=["feature_1", "feature_2", "abs_corr"])
research_assets["high_corr_pairs_abs_gt_0_8"] = high_corr_pairs
print("High-correlation pairs, abs(corr) >= 0.8:")
display(high_corr_pairs.head(50))

## SECTION 6 - Preprocessing Strategy

This section defines the modeling feature set and preprocessing principles:

- `sample_id` is never used as a feature.
- `property_organic_content` is never used inside test features.
- CatBoost receives categorical columns natively as strings.
- LightGBM/XGBoost receive fold-safe median imputation for numeric columns and fold-safe ordinal encoding for categorical columns.
- Constant columns may be dropped safely because they provide no split information.

In [ ]:
# Safe constant columns from train only. A column constant in train cannot provide useful supervised splits.
DROP_CONSTANT_COLUMNS = True
constant_cols_train = train_audit["constant_cols"]
constant_cols_train = [c for c in constant_cols_train if c in test_raw.columns]

base_drop_cols = [ID_COL, TARGET]
if DROP_CONSTANT_COLUMNS:
    base_drop_cols += constant_cols_train

base_feature_cols = [c for c in train_raw.columns if c not in base_drop_cols]
print("Base dropped columns:", base_drop_cols)
print("Number of base features:", len(base_feature_cols))

X_base = train_raw[base_feature_cols].copy()
y = train_raw[TARGET].copy()
X_test_base = test_raw[base_feature_cols].copy()

base_categorical_cols = X_base.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
base_numerical_cols = X_base.select_dtypes(include=[np.number]).columns.tolist()

print("Base categorical columns:", base_categorical_cols)
print("Base numerical columns:", base_numerical_cols)

assert ID_COL not in X_base.columns, "ID column leaked into features"
assert TARGET not in X_base.columns, "Target column leaked into features"
assert TARGET not in X_test_base.columns, "Target column leaked into test features"

## SECTION 7 - Feature Engineering

Feature engineering is performed on combined train-test features without using the target, preventing target leakage while keeping train and test transformations consistent.

Included features:

- Missing indicators and row-level missing counts
- Spectral block statistics for Band A and Band B
- Soil particle and cation chemistry ratios
- Geography bins and geography-combination categorical features
- Frequency encoding for important categorical columns

In [ ]:
def safe_divide(a, b):
    """Vectorized safe division that preserves NaN when division is not meaningful."""
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    result = a / b.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)


def add_if_columns_exist(df: pd.DataFrame, new_col: str, cols: list[str], operation) -> None:
    """Add engineered feature only if required source columns exist."""
    if all(c in df.columns for c in cols):
        df[new_col] = operation(df)


def create_features(df: pd.DataFrame, categorical_frequency_cols: Optional[list[str]] = None) -> pd.DataFrame:
    """Create leakage-safe features. This function must not use the target."""
    df = df.copy()
    original_cols = [c for c in df.columns if c not in [ID_COL, TARGET]]
    original_numeric_cols = df[original_cols].select_dtypes(include=[np.number]).columns.tolist()

    # -----------------------------
    # Missing indicators and counts
    # -----------------------------
    original_missing_mask = df[original_cols].isna()
    df["original_total_missing_count"] = original_missing_mask.sum(axis=1)
    df["original_numerical_missing_count"] = df[original_numeric_cols].isna().sum(axis=1) if original_numeric_cols else 0

    missing_indicator_cols = [c for c in original_cols if df[c].isna().any()]
    priority_missing_cols = [
        "property_acidity_index", "cation_Na", "latitude", "longitude",
        "cation_Ca", "cation_Mg", "cation_exchange_capacity",
    ]
    missing_indicator_cols = sorted(set(missing_indicator_cols).union([c for c in priority_missing_cols if c in df.columns]))
    for c in missing_indicator_cols:
        df[f"missing_{c}"] = df[c].isna().astype(int)

    # -----------------------------
    # Spectral block features
    # -----------------------------
    spectral_a_cols = [c for c in df.columns if c.startswith("spectral_band_A_PC_")]
    spectral_b_cols = [c for c in df.columns if c.startswith("spectral_band_B_PC_")]

    def add_spectral_stats(prefix: str, cols: list[str]) -> None:
        if not cols:
            return
        block = df[cols].apply(pd.to_numeric, errors="coerce")
        df[f"{prefix}_mean"] = block.mean(axis=1)
        df[f"{prefix}_std"] = block.std(axis=1)
        df[f"{prefix}_min"] = block.min(axis=1)
        df[f"{prefix}_max"] = block.max(axis=1)
        df[f"{prefix}_median"] = block.median(axis=1)
        df[f"{prefix}_abs_mean"] = block.abs().mean(axis=1)
        df[f"{prefix}_l2_norm"] = np.sqrt((block.fillna(0) ** 2).sum(axis=1))
        df[f"{prefix}_missing_count"] = block.isna().sum(axis=1)
        df[f"{prefix}_available_count"] = block.notna().sum(axis=1)

    add_spectral_stats("spectral_band_A", spectral_a_cols)
    add_spectral_stats("spectral_band_B", spectral_b_cols)
    df["missing_band_B"] = df[spectral_b_cols].isna().all(axis=1).astype(int) if spectral_b_cols else 0
    df["missing_band_A"] = df[spectral_a_cols].isna().all(axis=1).astype(int) if spectral_a_cols else 0

    # -----------------------------
    # Soil particle features
    # -----------------------------
    add_if_columns_exist(df, "particle_fine_minus_coarse", ["property_particle_fine", "property_particle_coarse"],
                         lambda x: x["property_particle_fine"] - x["property_particle_coarse"])
    add_if_columns_exist(df, "particle_fine_ratio", ["property_particle_fine", "property_particle_coarse"],
                         lambda x: safe_divide(x["property_particle_fine"], x["property_particle_fine"] + x["property_particle_coarse"]))
    add_if_columns_exist(df, "particle_coarse_ratio", ["property_particle_fine", "property_particle_coarse"],
                         lambda x: safe_divide(x["property_particle_coarse"], x["property_particle_fine"] + x["property_particle_coarse"]))

    # -----------------------------
    # Soil chemistry and cation features
    # -----------------------------
    cation_cols = [c for c in ["cation_Ca", "cation_Mg", "cation_Na"] if c in df.columns]
    if cation_cols:
        df["cation_total"] = df[cation_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)

    add_if_columns_exist(df, "Ca_Mg_ratio", ["cation_Ca", "cation_Mg"], lambda x: safe_divide(x["cation_Ca"], x["cation_Mg"]))
    add_if_columns_exist(df, "Ca_CEC_ratio", ["cation_Ca", "cation_exchange_capacity"], lambda x: safe_divide(x["cation_Ca"], x["cation_exchange_capacity"]))
    add_if_columns_exist(df, "Mg_CEC_ratio", ["cation_Mg", "cation_exchange_capacity"], lambda x: safe_divide(x["cation_Mg"], x["cation_exchange_capacity"]))
    add_if_columns_exist(df, "Na_CEC_ratio", ["cation_Na", "cation_exchange_capacity"], lambda x: safe_divide(x["cation_Na"], x["cation_exchange_capacity"]))
    add_if_columns_exist(df, "fine_x_CEC", ["property_particle_fine", "cation_exchange_capacity"], lambda x: x["property_particle_fine"] * x["cation_exchange_capacity"])
    add_if_columns_exist(df, "acidity_x_CEC", ["property_acidity_index", "cation_exchange_capacity"], lambda x: x["property_acidity_index"] * x["cation_exchange_capacity"])

    # -----------------------------
    # Geography features
    # -----------------------------
    if {"latitude", "longitude"}.issubset(df.columns):
        df["lat_lon_missing"] = (df["latitude"].isna() | df["longitude"].isna()).astype(int)
        df["lat_lon_interaction"] = pd.to_numeric(df["latitude"], errors="coerce") * pd.to_numeric(df["longitude"], errors="coerce")
        df["lat_abs"] = pd.to_numeric(df["latitude"], errors="coerce").abs()
        df["lon_abs"] = pd.to_numeric(df["longitude"], errors="coerce").abs()
        df["lat_bin"] = pd.cut(pd.to_numeric(df["latitude"], errors="coerce"), bins=10, labels=False, duplicates="drop")
        df["lon_bin"] = pd.cut(pd.to_numeric(df["longitude"], errors="coerce"), bins=10, labels=False, duplicates="drop")

    def combine_cat(new_col: str, cols: list[str]) -> None:
        if all(c in df.columns for c in cols):
            values = [df[c].astype(str).fillna("Missing") for c in cols]
            combined = values[0]
            for v in values[1:]:
                combined = combined + "__" + v
            df[new_col] = combined

    combine_cat("geo_macro_meso", ["geo_zone_macro", "geo_zone_meso"])
    combine_cat("geo_meso_micro", ["geo_zone_meso", "geo_zone_micro"])
    combine_cat("geo_macro_biome", ["geo_zone_macro", "biome"])
    combine_cat("biome_land_cover", ["biome", "land_cover_type"])
    combine_cat("rock_biome", ["parent_rock_type", "biome"])
    combine_cat("land_cover_geo_macro", ["land_cover_type", "geo_zone_macro"])

    # -----------------------------
    # Frequency encoding
    # -----------------------------
    categorical_frequency_cols = categorical_frequency_cols or []
    for c in categorical_frequency_cols:
        if c in df.columns:
            key = df[c].astype(str).fillna("Missing")
            freq = key.value_counts(dropna=False, normalize=True)
            count = key.value_counts(dropna=False)
            df[f"{c}_freq"] = key.map(freq).astype(float)
            df[f"{c}_count"] = key.map(count).astype(float)

    # Clean infinities
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

# Feature engineering is fit on combined train+test features only, without target.
combined_features = pd.concat([X_base, X_test_base], axis=0, ignore_index=True)
frequency_cols = [
    "source_id", "geo_zone_macro", "geo_zone_meso", "geo_zone_micro",
    "land_cover_type", "biome", "parent_rock_type", "sampling_strategy",
    "has_band_A_spectrum", "has_band_B_spectrum",
]
frequency_cols = [c for c in frequency_cols if c in combined_features.columns]

combined_fe = create_features(combined_features, categorical_frequency_cols=frequency_cols)
X_fe = combined_fe.iloc[:len(train_raw)].reset_index(drop=True)
X_test_fe = combined_fe.iloc[len(train_raw):].reset_index(drop=True)

# Final feature columns
feature_cols = X_fe.columns.tolist()
cat_features = X_fe.select_dtypes(include=["object", "category"]).columns.tolist()
num_features = [c for c in feature_cols if c not in cat_features]

print("Engineered train shape:", X_fe.shape)
print("Engineered test shape:", X_test_fe.shape)
print("Categorical features after FE:", len(cat_features))
print("Numerical features after FE:", len(num_features))
print("New feature count:", X_fe.shape[1] - X_base.shape[1])

assert X_fe.shape[0] == len(train_raw)
assert X_test_fe.shape[0] == len(test_raw)
assert ID_COL not in X_fe.columns
assert TARGET not in X_fe.columns

## SECTION 8 - Validation Strategy

The main validation uses `StratifiedKFold` adapted for regression by binning the target with quantiles. This helps keep the target distribution balanced across folds.

Each model returns:

- Out-of-fold predictions
- Averaged test predictions
- Fold RMSE scores
- Mean and standard deviation CV RMSE
- Train RMSE per fold for overfit/underfit analysis
- Trained fold models

In [ ]:
def make_regression_stratified_folds(y: pd.Series, n_splits: int = N_SPLITS, seed: int = RANDOM_STATE):
    """Create stratified folds for regression using quantile-binned targets."""
    y = pd.Series(y).reset_index(drop=True)
    n_bins = min(10, max(2, len(y) // n_splits))
    try:
        y_bins = pd.qcut(y, q=n_bins, labels=False, duplicates="drop")
        if pd.Series(y_bins).nunique() < 2:
            raise ValueError("Not enough unique target bins")
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = list(splitter.split(np.zeros(len(y)), y_bins))
        strategy = "StratifiedKFold with target quantile bins"
    except Exception as exc:
        print(f"Falling back to KFold because target binning failed: {exc}")
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = list(splitter.split(np.zeros(len(y))))
        strategy = "KFold fallback"
    return folds, strategy

folds, validation_strategy = make_regression_stratified_folds(y, N_SPLITS, RANDOM_STATE)
print("Validation strategy:", validation_strategy)

# Check target balance per fold
fold_balance = []
for fold, (_, val_idx) in enumerate(folds, 1):
    yy = y.iloc[val_idx]
    fold_balance.append({
        "fold": fold,
        "n_valid": len(val_idx),
        "target_mean": yy.mean(),
        "target_median": yy.median(),
        "target_min": yy.min(),
        "target_max": yy.max(),
        "target_std": yy.std(),
    })
fold_balance_df = pd.DataFrame(fold_balance)
display(fold_balance_df)


def prepare_catboost_frame(X: pd.DataFrame, cat_cols: list[str]) -> pd.DataFrame:
    """Prepare a DataFrame for CatBoost native categorical handling."""
    X_out = X.copy()
    for c in cat_cols:
        if c in X_out.columns:
            X_out[c] = X_out[c].astype(str).fillna("Missing")
    return X_out




def make_median_imputer():
    """Create a median imputer while preserving all-empty features when sklearn supports it."""
    try:
        return SimpleImputer(strategy="median", keep_empty_features=True)
    except TypeError:
        return SimpleImputer(strategy="median")

def prepare_encoded_fold(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
):
    """Fold-safe preprocessing for LightGBM/XGBoost/Ridge: median impute numeric and ordinal encode categorical."""
    Xtr_parts, Xva_parts, Xte_parts, feature_names = [], [], [], []

    if num_cols:
        imputer = make_median_imputer()
        Xtr_num = pd.DataFrame(imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
        Xva_num = pd.DataFrame(imputer.transform(X_valid[num_cols]), columns=num_cols, index=X_valid.index)
        Xte_num = pd.DataFrame(imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)
        Xtr_parts.append(Xtr_num)
        Xva_parts.append(Xva_num)
        Xte_parts.append(Xte_num)
        feature_names.extend(num_cols)

    if cat_cols:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        Xtr_cat_raw = X_train[cat_cols].astype(str).fillna("Missing")
        Xva_cat_raw = X_valid[cat_cols].astype(str).fillna("Missing")
        Xte_cat_raw = X_test[cat_cols].astype(str).fillna("Missing")
        encoded_cat_cols = [f"{c}__ord" for c in cat_cols]
        Xtr_cat = pd.DataFrame(encoder.fit_transform(Xtr_cat_raw), columns=encoded_cat_cols, index=X_train.index)
        Xva_cat = pd.DataFrame(encoder.transform(Xva_cat_raw), columns=encoded_cat_cols, index=X_valid.index)
        Xte_cat = pd.DataFrame(encoder.transform(Xte_cat_raw), columns=encoded_cat_cols, index=X_test.index)
        Xtr_parts.append(Xtr_cat)
        Xva_parts.append(Xva_cat)
        Xte_parts.append(Xte_cat)
        feature_names.extend(encoded_cat_cols)

    Xtr = pd.concat(Xtr_parts, axis=1)
    Xva = pd.concat(Xva_parts, axis=1)
    Xte = pd.concat(Xte_parts, axis=1)
    return Xtr, Xva, Xte, feature_names

experiment_log = []
model_outputs = {}

## SECTION 9 - Baseline Model

Baselines provide a reference point for the main models.

Included baselines:

1. Mean target baseline
2. Ridge regression baseline with fold-safe preprocessing
3. HistGradientBoosting baseline with fold-safe preprocessing

In [ ]:
def log_experiment(model_name: str, features: str, fold_scores: list[float], notes: str, train_scores: Optional[list[float]] = None):
    row = {
        "model_name": model_name,
        "features": features,
        "mean_cv_rmse": float(np.mean(fold_scores)),
        "std_cv_rmse": float(np.std(fold_scores)),
        "fold_scores": fold_scores,
        "notes": notes,
    }
    if train_scores is not None:
        row["mean_train_rmse"] = float(np.mean(train_scores))
        row["train_valid_gap"] = float(np.mean(fold_scores) - np.mean(train_scores))
    experiment_log.append(row)
    return row

# Baseline 1: mean prediction
mean_oof = np.zeros(len(y))
mean_test_pred = np.zeros(len(test_raw))
mean_fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    pred_value = y.iloc[tr_idx].mean()
    mean_oof[val_idx] = pred_value
    fold_rmse = rmse(y.iloc[val_idx], mean_oof[val_idx])
    mean_fold_scores.append(fold_rmse)
mean_test_pred[:] = y.mean()
log_experiment("Mean Baseline", "target mean only", mean_fold_scores, "Predicts each fold training target mean")
print("Mean baseline RMSE:", np.mean(mean_fold_scores), "+/-", np.std(mean_fold_scores))

model_outputs["mean_baseline"] = {
    "oof": mean_oof,
    "test_pred": mean_test_pred,
    "scores": mean_fold_scores,
}

In [ ]:
def run_ridge_baseline(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=10.0, random_state=RANDOM_STATE)),
        ])
        model.fit(X_tr, y_tr)
        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        print(f"Ridge fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    return {"oof": oof, "test_pred": test_pred, "scores": fold_scores, "train_scores": train_scores}

ridge_output = run_ridge_baseline(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("Ridge Baseline", "all engineered features, encoded", ridge_output["scores"], "Linear baseline with median imputation, ordinal encoding, scaling", ridge_output["train_scores"])
model_outputs["ridge"] = ridge_output
print("Ridge baseline RMSE:", np.mean(ridge_output["scores"]), "+/-", np.std(ridge_output["scores"]))

In [ ]:
def run_hgb_baseline(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        model = HistGradientBoostingRegressor(
            learning_rate=0.04,
            max_iter=300 if FAST_MODE else 800,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=RANDOM_STATE,
            early_stopping=True,
        )
        model.fit(X_tr, y_tr)
        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        print(f"HGB fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    return {"oof": oof, "test_pred": test_pred, "scores": fold_scores, "train_scores": train_scores}

hgb_output = run_hgb_baseline(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("HistGradientBoosting Baseline", "all engineered features, encoded", hgb_output["scores"], "Tree baseline with fold-safe preprocessing", hgb_output["train_scores"])
model_outputs["hgb"] = hgb_output
print("HGB baseline RMSE:", np.mean(hgb_output["scores"]), "+/-", np.std(hgb_output["scores"]))

## SECTION 10 - Main Models: CatBoost, LightGBM, XGBoost

Three main models are trained using the same fold split:

1. CatBoostRegressor: native categorical handling
2. LightGBMRegressor: fold-safe numeric median imputation and categorical ordinal encoding
3. XGBoostRegressor: same fold-safe encoded data as LightGBM

In [ ]:
def run_catboost_cv(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols: list[str]):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []
    models = []
    feature_importance_rows = []

    X_cb_full = prepare_catboost_frame(X, cat_cols)
    X_test_cb = prepare_catboost_frame(X_test, cat_cols)

    cat_indices = [X_cb_full.columns.get_loc(c) for c in cat_cols if c in X_cb_full.columns]

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr, X_val = X_cb_full.iloc[tr_idx], X_cb_full.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=MAIN_N_ESTIMATORS,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5,
            random_strength=1.0,
            bagging_temperature=0.5,
            random_seed=RANDOM_STATE + fold,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=200,
            allow_writing_files=False,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_indices,
            use_best_model=True,
        )

        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_test_cb)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        models.append(model)

        fi = pd.DataFrame({
            "feature": X_cb_full.columns,
            "importance": model.get_feature_importance(),
            "fold": fold,
        })
        feature_importance_rows.append(fi)
        print(f"CatBoost fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    feature_importance = pd.concat(feature_importance_rows, ignore_index=True)
    return {
        "oof": oof,
        "test_pred": test_pred,
        "scores": fold_scores,
        "train_scores": train_scores,
        "models": models,
        "feature_importance": feature_importance,
    }

catboost_output = run_catboost_cv(X_fe, X_test_fe, y, folds, cat_features)
log_experiment("CatBoostRegressor", "all engineered features, native categorical", catboost_output["scores"], "Main model with native categorical and missing handling", catboost_output["train_scores"])
model_outputs["catboost"] = catboost_output
print("CatBoost CV RMSE:", np.mean(catboost_output["scores"]), "+/-", np.std(catboost_output["scores"]))

In [ ]:
def fit_lgbm_compatible(model, X_tr, y_tr, X_val, y_val):
    """Fit LightGBM across multiple API versions."""
    try:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(200)],
        )
    except TypeError:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=200,
        )
    return model


def run_lgbm_cv(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []
    models = []
    feature_importance_rows = []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=MAIN_N_ESTIMATORS,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            min_child_samples=20,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            verbosity=-1,
        )
        model = fit_lgbm_compatible(model, X_tr, y_tr, X_val, y_val)

        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        models.append(model)

        fi = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_,
            "fold": fold,
        })
        feature_importance_rows.append(fi)
        print(f"LightGBM fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    feature_importance = pd.concat(feature_importance_rows, ignore_index=True)
    return {
        "oof": oof,
        "test_pred": test_pred,
        "scores": fold_scores,
        "train_scores": train_scores,
        "models": models,
        "feature_importance": feature_importance,
    }

lgbm_output = run_lgbm_cv(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("LightGBMRegressor", "all engineered features, encoded", lgbm_output["scores"], "Main model with fold-safe imputation and ordinal encoding", lgbm_output["train_scores"])
model_outputs["lightgbm"] = lgbm_output
print("LightGBM CV RMSE:", np.mean(lgbm_output["scores"]), "+/-", np.std(lgbm_output["scores"]))

In [ ]:
def fit_xgb_compatible(params: dict, X_tr, y_tr, X_val, y_val):
    """Fit XGBoost across old/new API differences for early stopping."""
    params = params.copy()
    try:
        model = XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)
    except TypeError:
        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=200)
    return model


def run_xgb_cv(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []
    models = []
    feature_importance_rows = []

    params = dict(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=MAIN_N_ESTIMATORS,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        tree_method="hist",
        n_jobs=-1,
    )

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        params["random_state"] = RANDOM_STATE + fold
        model = fit_xgb_compatible(params, X_tr, y_tr, X_val, y_val)

        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        models.append(model)

        fi = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_,
            "fold": fold,
        })
        feature_importance_rows.append(fi)
        print(f"XGBoost fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    feature_importance = pd.concat(feature_importance_rows, ignore_index=True)
    return {
        "oof": oof,
        "test_pred": test_pred,
        "scores": fold_scores,
        "train_scores": train_scores,
        "models": models,
        "feature_importance": feature_importance,
    }

xgb_output = run_xgb_cv(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("XGBoostRegressor", "all engineered features, encoded", xgb_output["scores"], "Main model with fold-safe imputation and ordinal encoding", xgb_output["train_scores"])
model_outputs["xgboost"] = xgb_output
print("XGBoost CV RMSE:", np.mean(xgb_output["scores"]), "+/-", np.std(xgb_output["scores"]))

## SECTION 11 - Ensemble, Model Selection, and Final Prediction

The final Kaggle submission uses out-of-fold validated ensemble predictions.

Ensemble strategies:

1. Equal weight: CatBoost + LightGBM + XGBoost
2. Fixed weighted ensemble: 0.50 CatBoost + 0.30 LightGBM + 0.20 XGBoost
3. Simple OOF weight search with step 0.05

The best OOF RMSE ensemble is used to generate `submission.csv`.

In [ ]:
experiment_log_df = pd.DataFrame(experiment_log).sort_values("mean_cv_rmse")
print("Experiment log:")
display(experiment_log_df)

main_model_names = ["catboost", "lightgbm", "xgboost"]
assert all(name in model_outputs for name in main_model_names), "All three main models must be trained before ensembling."

main_oofs = {name: model_outputs[name]["oof"] for name in main_model_names}
main_test_preds = {name: model_outputs[name]["test_pred"] for name in main_model_names}


def weighted_average(pred_dict: dict[str, np.ndarray], weights: dict[str, float]) -> np.ndarray:
    pred = None
    for name, w in weights.items():
        if pred is None:
            pred = w * pred_dict[name]
        else:
            pred += w * pred_dict[name]
    return pred

ensemble_rows = []

# 1. Equal weight
weights_equal = {name: 1 / len(main_model_names) for name in main_model_names}
oof_equal = weighted_average(main_oofs, weights_equal)
ensemble_rows.append({"ensemble_name": "equal_weight", "weights": weights_equal, "oof_rmse": rmse(y, oof_equal)})

# 2. Recommended fixed weighted ensemble
weights_fixed = {"catboost": 0.50, "lightgbm": 0.30, "xgboost": 0.20}
oof_fixed = weighted_average(main_oofs, weights_fixed)
ensemble_rows.append({"ensemble_name": "fixed_0.50_0.30_0.20", "weights": weights_fixed, "oof_rmse": rmse(y, oof_fixed)})

# 3. Simple weight search, step 0.05
step = 0.05
values = np.round(np.arange(0, 1 + step, step), 2)
best = {"ensemble_name": "search_weight", "weights": None, "oof_rmse": np.inf}
for w_cat in values:
    for w_lgb in values:
        w_xgb = round(1.0 - w_cat - w_lgb, 2)
        if w_xgb < 0 or w_xgb > 1:
            continue
        weights = {"catboost": float(w_cat), "lightgbm": float(w_lgb), "xgboost": float(w_xgb)}
        oof_pred = weighted_average(main_oofs, weights)
        score = rmse(y, oof_pred)
        if score < best["oof_rmse"]:
            best = {"ensemble_name": "search_weight", "weights": weights, "oof_rmse": score}
ensemble_rows.append(best)

ensemble_results = pd.DataFrame(ensemble_rows).sort_values("oof_rmse")
print("Ensemble results:")
display(ensemble_results)

best_ensemble = ensemble_results.iloc[0]
best_weights = best_ensemble["weights"]
print("Selected ensemble:", best_ensemble["ensemble_name"])
print("Selected weights:", best_weights)
print("Selected OOF RMSE:", best_ensemble["oof_rmse"])

final_oof_pred = weighted_average(main_oofs, best_weights)
final_test_pred = weighted_average(main_test_preds, best_weights)
final_test_pred = np.clip(final_test_pred, 0, None)
final_oof_pred_clipped = np.clip(final_oof_pred, 0, None)

print("Final OOF RMSE before clipping:", rmse(y, final_oof_pred))
print("Final OOF RMSE after clipping:", rmse(y, final_oof_pred_clipped))
print("Final test prediction summary:")
display(pd.Series(final_test_pred, name=TARGET).describe().to_frame().T)

# Create submission following sample_submission order exactly
prediction_frame = pd.DataFrame({ID_COL: test_raw[ID_COL].values, TARGET: final_test_pred})
submission = sample_submission[[ID_COL]].merge(prediction_frame, on=ID_COL, how="left")

# Strict final validation
assert list(submission.columns) == [ID_COL, TARGET], f"Submission columns must be {[ID_COL, TARGET]}"
assert len(submission) == len(sample_submission), "Submission row count must match sample_submission"
assert submission[ID_COL].equals(sample_submission[ID_COL]), "Submission ID order must follow sample_submission"
assert submission[TARGET].notna().all(), "Submission predictions must not contain NaN"
assert np.isfinite(submission[TARGET]).all(), "Submission predictions must be finite"
assert (submission[TARGET] >= 0).all(), "Submission predictions must not be negative"

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())
display(submission.shape)
display(submission[TARGET].describe().to_frame().T)

# Optional Colab download
try:
    from google.colab import files
    files.download("submission.csv")
except Exception:
    pass

## SECTION 12 - Final Reporting Assets for Research Questions

This section builds the evidence assets needed for final written answers. The final answers should be written after inspecting these outputs, not guessed.

Assets included:

- Final experiment log
- OOF prediction vs actual
- Residual distribution
- Residual by target range
- Feature importance tables for CatBoost, LightGBM, and XGBoost
- Fair feature-engineering impact comparison
- Geographic/ecosystem summaries
- Research question answer template

In [ ]:
# Final experiment table including ensemble
ensemble_log_row = {
    "model_name": f"Final Ensemble - {best_ensemble['ensemble_name']}",
    "features": "all engineered features",
    "mean_cv_rmse": float(best_ensemble["oof_rmse"]),
    "std_cv_rmse": np.nan,
    "fold_scores": None,
    "notes": f"Weights: {best_weights}",
    "mean_train_rmse": np.nan,
    "train_valid_gap": np.nan,
}
final_experiment_log = pd.concat([experiment_log_df, pd.DataFrame([ensemble_log_row])], ignore_index=True).sort_values("mean_cv_rmse")
print("Final experiment log:")
display(final_experiment_log)

# Residual assets
residuals = y.values - final_oof_pred
residual_df = pd.DataFrame({
    TARGET: y.values,
    "oof_prediction": final_oof_pred,
    "oof_prediction_clipped": final_oof_pred_clipped,
    "residual": residuals,
    "abs_residual": np.abs(residuals),
})
residual_df["target_bin"] = pd.qcut(residual_df[TARGET], q=5, duplicates="drop")
print("Residual summary:")
display(residual_df.describe())
print("Residual by target bin:")
display(residual_df.groupby("target_bin", observed=False)["abs_residual"].agg(["count", "mean", "median", "std", "max"]))

plt.figure(figsize=(6, 6))
plt.scatter(residual_df[TARGET], residual_df["oof_prediction"], alpha=0.4)
lims = [min(residual_df[TARGET].min(), residual_df["oof_prediction"].min()), max(residual_df[TARGET].max(), residual_df["oof_prediction"].max())]
plt.plot(lims, lims)
plt.xlabel("Actual")
plt.ylabel("OOF Prediction")
plt.title("OOF Prediction vs Actual")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(residual_df["residual"], bins=50)
plt.xlabel("Residual = actual - prediction")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.scatter(residual_df[TARGET], residual_df["residual"], alpha=0.4)
plt.axhline(0)
plt.xlabel("Actual Target")
plt.ylabel("Residual")
plt.title("Residual vs Actual Target")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance summaries
def summarize_importance(fi_df: pd.DataFrame, top_n: int = 30) -> pd.DataFrame:
    if fi_df is None or fi_df.empty:
        return pd.DataFrame()
    return (
        fi_df.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
        .head(top_n)
    )

importance_assets = {}
for name in ["catboost", "lightgbm", "xgboost"]:
    fi = summarize_importance(model_outputs[name].get("feature_importance"), top_n=30)
    importance_assets[name] = fi
    print(f"Top feature importance - {name}:")
    display(fi)
    if not fi.empty:
        plt.figure(figsize=(8, 6))
        plt.barh(fi["feature"][::-1], fi["importance"][::-1])
        plt.xlabel("Mean Importance")
        plt.title(f"Top Feature Importance - {name}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Fair feature engineering impact comparison: same model family, raw vs engineered features.
# Uses a lightweight LightGBM setting so it does not dominate runtime.

def quick_lgbm_cv_score(X: pd.DataFrame, X_test_unused: pd.DataFrame, y: pd.Series, folds, label: str):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    oof = np.zeros(len(y))
    scores = []
    dummy_test = X_test_unused.copy()
    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, _, _ = prepare_encoded_fold(X_tr_raw, X_val_raw, dummy_test, cat_cols, num_cols)
        model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=300 if FAST_MODE else 800,
            learning_rate=0.04,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            verbosity=-1,
        )
        model = fit_lgbm_compatible(model, X_tr, y_tr, X_val, y_val)
        val_pred = model.predict(X_val)
        oof[val_idx] = val_pred
        scores.append(rmse(y_val, val_pred))
    return {"label": label, "oof_rmse": rmse(y, oof), "mean_fold_rmse": float(np.mean(scores)), "std_fold_rmse": float(np.std(scores))}

raw_lgb_score = quick_lgbm_cv_score(X_base.reset_index(drop=True), X_test_base.reset_index(drop=True), y.reset_index(drop=True), folds, "LightGBM raw features")
fe_lgb_score = quick_lgbm_cv_score(X_fe.reset_index(drop=True), X_test_fe.reset_index(drop=True), y.reset_index(drop=True), folds, "LightGBM engineered features")
fe_impact_df = pd.DataFrame([raw_lgb_score, fe_lgb_score])
fe_impact_df["improvement_vs_raw"] = raw_lgb_score["oof_rmse"] - fe_impact_df["oof_rmse"]
print("Fair feature engineering impact comparison:")
display(fe_impact_df)

### Research Question Answer Template

Use the following template after reviewing the generated outputs:

1. **Urgency of predicting soil organic content**  
   Evidence to use: business/scientific context, target distribution, potential benefit of rapid estimation.

2. **Overfit or underfit analysis**  
   Evidence to use: mean train RMSE, mean validation RMSE, train-valid gap, fold stability, residual plots.

3. **Organic content distribution across geographic zones**  
   Evidence to use: `target_by_geo_zone_macro`, `target_by_geo_zone_meso`, `target_by_geo_zone_micro`.

4. **Correlation/pattern across geographic hierarchy**  
   Evidence to use: geographic grouped means/medians/counts and extreme-zone comparisons.

5. **Ecosystem averages under low acidity and low CEC**  
   Evidence to use: `low_acidity_low_cec` filtered summaries by biome, land cover, and macro zone.

6. **Outliers from land cover and geography combinations**  
   Evidence to use: `land_cover_geo_outliers`, especially groups with enough sample count.

7. **Multicollinearity and high-correlation features**  
   Evidence to use: `high_corr_pairs_abs_gt_0_8`, correlation heatmap, and model type discussion.

8. **Feature engineering explanation**  
   Evidence to use: engineered feature list, fair raw-vs-FE comparison, feature importance.

9. **Model choice and performance**  
   Evidence to use: final experiment log, CatBoost/LightGBM/XGBoost CV RMSE, final ensemble RMSE.

10. **RMSE metric and possible external data**  
   External data was not used in the modeling pipeline or final Kaggle submission because the competition rules prohibit any data not officially provided. The final model only uses features derived from train.csv and test.csv.

   For discussion purposes only, external sources such as climate data, elevation, rainfall, or soil maps could theoretically improve prediction if allowed. However, these sources were not used, downloaded, merged, or integrated into the submitted model.

In [ ]:
# Save useful outputs for reporting
Path("outputs").mkdir(exist_ok=True)
final_experiment_log.to_csv("outputs/final_experiment_log.csv", index=False)
ensemble_results.to_csv("outputs/ensemble_results.csv", index=False)
residual_df.to_csv("outputs/oof_residuals.csv", index=False)
fe_impact_df.to_csv("outputs/feature_engineering_impact.csv", index=False)

for key, value in research_assets.items():
    if isinstance(value, pd.DataFrame):
        safe_key = key.replace("/", "_").replace(" ", "_")
        value.to_csv(f"outputs/{safe_key}.csv", index=False)

for name, fi in importance_assets.items():
    if isinstance(fi, pd.DataFrame) and not fi.empty:
        fi.to_csv(f"outputs/feature_importance_{name}.csv", index=False)

print("Saved reporting assets to outputs/ folder")
print("Final notebook workflow complete. Ready to submit submission.csv to Kaggle.")